> **Historical notebook.** This notebook is preserved for provenance. Cell outputs have been cleared, and the competition data is not included because Zindi's rules prohibit redistribution. See the repository README for data setup; the maintained entry point is `train.py` at the repository root.

In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_row', 1000)
df = pd.read_csv('../data/Train.csv')
df.head()

In [ ]:
train = df.iloc[:, :-1]
test = pd.read_csv('../data/Test.csv')
target = df.iloc[:, -1]

In [ ]:
print(train.shape)
print(test.shape)
print(target.shape)

In [ ]:
text = pd.concat([train, test], ignore_index=True)
text.shape

In [ ]:
import re
import nltk
nltk.download('stopwords')

In [ ]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [ ]:
corpus = []
for i in range(0, 925):
    feeling = re.sub('[^a-zA-Z]', ' ', str(text['text'][i]))
    feeling = feeling.lower()
    feeling = feeling.split()
    ps = PorterStemmer()
    feeling = [ps.stem(word) for word in feeling if not word in set(stopwords.words('english'))]
    feeling = ' '.join(feeling)
    corpus.append(feeling)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
result = cv.fit_transform(corpus).toarray()

In [ ]:
result.shape

In [ ]:
dataframe = pd.DataFrame(result)
training = df.iloc[:,0].to_frame()
training = training.join(dataframe.iloc[:616,:])
training = training.join(target)
training.head()

In [ ]:
training.shape

In [ ]:
testing = test.iloc[:,0].to_frame()
_test = dataframe.iloc[616:,:].reset_index()
_test.drop(columns = 'index', inplace = True)
testing = testing.join(_test)
testing.head()

In [ ]:
training['label']=training['label'].astype('category')
training['label']=training['label'].cat.codes
X = training.iloc[:,1:779].values
y = training.iloc[:,-1].values

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

y = to_categorical(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, random_state = 0)

classifier = Sequential()

In [ ]:
# Adding the input layer and the first hidden layer
classifier.add(Dense(units = 389, kernel_initializer = 'uniform', activation = 'relu', input_dim = 778))

# Adding the second hidden layer
classifier.add(Dense(units = 389, kernel_initializer = 'uniform', activation = 'relu'))

# Adding the output layer
classifier.add(Dense(units = 4, kernel_initializer = 'uniform', activation = 'softmax'))

# Compiling the ANN
classifier.compile(optimizer = 'rmsprop', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# Fitting the ANN to the Training set
classifier.fit(X_train, y_train, batch_size = 10, epochs = 100)

In [ ]:
import numpy as np
y_pred = classifier.predict_proba(X_test)
np.around(y_pred, decimals=2)

In [ ]:
y_test

In [ ]:
Sub= testing.iloc[:,1:779].values
preds = classifier.predict_proba(Sub)
preds = np.around(preds, decimals=3)

In [ ]:
Submission = testing.iloc[:, 0].to_frame()
 # Depression,Alcohol,Suicide,Drugs
Submission['Depression']=preds[:,1]
Submission['Alcohol']=preds[:,0]
Submission['Drugs']=preds[:,2]
Submission['Suicide']=preds[:,3]
Submission.head(10)

In [ ]:
test.head(10)

In [ ]:
Submission.to_csv('../submissions/Submission_v5.csv', index=False)